# CattleClass — Google Colab Training Notebook

This notebook reproduces the locked CID benchmark and builds a TensorFlow/Keras research demo that shows breed, age and weight from one photo.

Use a Python 3.12 runtime. Select a GPU in Runtime → Change runtime type for training. Mount Drive to keep checkpoints across resets.


## 1. Mount Google Drive
Drive stores the working data and checkpoints so they survive a Colab runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%env CATTLE_WORK_DIR=/content/drive/MyDrive/CattleClass-work

## 2. Clone the project and install dependencies
Run this the first time in a fresh runtime. If the folder already exists, use the update cell below instead.

In [ ]:
%cd /content
!git clone -b cattleclass-project https://github.com/ShayanAhmed12/CattleClass.git
%cd /content/CattleClass
%pip install -r requirements.txt

### Update an existing checkout
Use this cell instead of cloning again if `/content/CattleClass` already exists.

In [ ]:
!git -C /content/CattleClass pull --ff-only origin cattleclass-project
%cd /content/CattleClass
%pip install -r requirements.txt


## 3. Check the runtime
Confirm that TensorFlow can see the GPU before training.

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 4. Prepare the dataset

Preparation checks CID against the repository's locked animal-grouped folds. If existing models were made using another split, the next training step will rebuild incompatible checkpoints.


In [ ]:
%cd /content/CattleClass
!python prepare_data.py

## 5. Train the five breed folds
This trains the breed-classification folds. Completed compatible checkpoints may be skipped by the project training script.

In [ ]:
!python train.py

## 6. Fit the final breed model
The project guide indicates that this saves `models/final.keras` under the configured work directory.

In [ ]:
!python train.py --final

## 7. Optional breed-only demo

Open the link in the cell output. Stop the demo cell before continuing to training or evaluation.


In [ ]:
!python demo.py --share

### Alternative: predict directly in the notebook
Use this if the Gradio share link does not work. Upload a clear photo of one animal.

In [ ]:
from google.colab import files
from demo import predict

uploaded = files.upload()
if uploaded:
    image_path = next(iter(uploaded))
    print(predict(image_path))

## 8. Evaluate breed predictions
After the five folds finish, generate evaluation metrics, robustness plots, and example prediction visualizations.

In [ ]:
!python evaluate.py --robustness --figures

## 9. Train the traits and build the three-output research model

Run after Sections 4–6. The commands train, score and package the experimental weight and age models. The joint report scores breed, age and weight on the same held-out CID animals.


In [ ]:
!python train.py --traits && python evaluate.py --traits && python -m cattle_breed.research


### Launch the three-output research demo
Open the Gradio link and upload one cattle photo. Age is a choice between the 24- and 30-month CID labels. Weight is shown in kilograms. The research outputs use CID labels and lack independent field validation. Stop the demo cell when done.


In [ ]:
!python demo.py --research --share


## 10. Inspect generated results
Check the output files before deciding what to commit. Do not commit datasets, credentials, temporary files, or large checkpoints unless the team explicitly wants them in Git.

In [ ]:
!find results -maxdepth 3 -type f -print 2>/dev/null | head -200
!git status --short

## 11. Resume after a runtime reset

1. Reconnect to a GPU runtime.
2. Rerun Sections 1 and 2, using the same Drive work directory.
3. If data preparation already completed, continue from training.
4. Run evaluation after training completes.
5. Check `results/` and push only the intended outputs.

**Tip:** Keep the runtime connected while training or while using the Gradio demo. The demo share link stops when its cell/runtime stops.
